# 02 · Auditoría de conteos y puerta P1 — Mortalidad neonatal NCHS (M4)

| Campo | Contenido |
|---|---|
| Secuencia del framework | 01: E1 · E2 · E3 → **02: E4 · P1** → 03: E5 · E6 · E7 · P2 · P3 → 04: E8 → 05: E8 · E9 · P4 → 06: E10 · E11 |
| Qué hace | Cuenta población y eventos por etiqueta y cohorte, revisa la codificación de la edad al morir, las frecuencias y los desconocidos de cada predictor candidato en toda la población (nunca separados por desenlace), y verifica que los conteos de referencia se reproduzcan |
| Etapas y puertas | E4 · P1 · verificación previa de P2 (desconocidos) |
| Decisiones que originó | M4-D1 (punto de referencia a las 48 h), M4-D2 (sin estado civil), M4-D3 (imputación sin indicadores de faltante); ver la ficha del desenlace |
| Entradas | `data/interim/cohort_<año>.parquet` (notebook 01) |
| Salidas | `data/outputs_02_count_audit.txt` (solo conteos agregados; de 1 a 9 se muestran como "<10") |
| Tiempo y memoria | Unos minutos; lee un campo a la vez (≈1 GB) |
| Anterior / siguiente | `01_build_cohorts` / `03_predictors_g2_g3` |

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import COHORTS, OUT_DIR

# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")


OUT_FILE = ROOT / "data" / "outputs_02_count_audit.txt"
# Gate G1: reference counts of the data adapter (data/outputs_01_build_cohorts.txt):
# (population of 22-47 weeks, neonatal deaths 0-27 days)
EXPECTED = {2016: (3938288, 11497), 2017: (3848558, 11057), 2018: (3785243, 10703),
            2019: (3741187, 10249), 2020: (3607479, 9574), 2021: (3657762, 9633),
            2022: (3661524, 9967), 2023: (3589826, 9879)}
# "Unknown or not stated" codes (user guides). Character fields use "U"; DMAR is blank when the
# state does not report marital status. "Not applicable" codes (ILP_R 888, ME_TRIAL "X") are
# categories, not unknowns.
UNKNOWN = {"MRACEHISP": [8], "MEDUC": [9], "MBSTATE_REC": [3], "PAY_REC": [9], "BFACIL3": [9],
           "PRIORLIVE": [99], "PRIORDEAD": [99], "PRIORTERM": [99], "ILP_R": [999],
           "RF_CESARN": [99], "PRECARE": [99], "PREVIS": [99], "BMI": [99.9], "WTGAIN": [99],
           "ME_PRES": [9], "ME_ROUT": [9], "ATTEND": [9], "APGAR5": [99], "BRTHWGT": [9999]}
CHAR_UNKNOWN = "U"
BLOCKS = {
    "mother": ["MAGER", "MRACEHISP", "MEDUC", "DMAR", "MBSTATE_REC", "PAY_REC", "WIC"],
    "obstetric_history": ["PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESAR", "RF_CESARN", "RF_PPB"],
    "current_pregnancy": ["PRECARE", "PREVIS", "CIG_REC", "BMI", "WTGAIN", "RF_PDIAB", "RF_GDIAB",
                          "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "IP_GON", "IP_SYPH",
                          "IP_CHLAM", "IP_HEPB", "IP_HEPC"],
    "labor_delivery": ["LD_INDL", "LD_AUGM", "LD_STER", "LD_ANTB", "LD_CHOR", "LD_ANES", "ME_PRES",
                       "ME_ROUT", "ME_TRIAL", "BFACIL3", "ATTEND"],
    "newborn": ["DPLURAL", "SEX", "COMBGEST", "BRTHWGT", "APGAR5", "CA_ANEN", "CA_MNSB", "CA_CCHD",
                "CA_CDH", "CA_OMPH", "CA_GAST", "CA_LIMB", "CA_CLEFT", "CA_CLPAL", "CA_DOWN",
                "CA_DISOR", "CA_HYPO"],
}
FIELDS = [c for b in BLOCKS.values() for c in b]
NUMERIC = ["MAGER", "PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESARN", "PRECARE", "PREVIS",
           "BMI", "WTGAIN", "COMBGEST", "BRTHWGT", "APGAR5"]
CATEGORICAL = [c for c in FIELDS if c not in NUMERIC]
LABELS = ["label_main", "died_first_24h", "label_day1_27", "label_early_neonatal"]
DEATH = ["died_infant", "d_age_days", "d_ager5"]
lines = []

def show(title, obj):
    """Print and keep an aggregate table; counts from 1 to 9 are suppressed as '<10'."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        sup = lambda v: ("<10" if isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_))
                         and 0 < v < 10 else v)
        txt = (obj.map(sup) if isinstance(obj, pd.DataFrame) else obj.map(sup)).to_string()
    else:
        txt = str(obj)
    block = f"\n== {title}\n{txt}"
    print(block)
    lines.append(block)

SUPPRESS_SMALL = True
import contextlib, io, html as _html
_show_en = show

def _fmt(v):
    # Display format of a table cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if SUPPRESS_SMALL and isinstance(v, (int, np.integer)) and 0 < v < 10:
        return "<10"
    if isinstance(v, (int, np.integer)) or (isinstance(v, (float, np.floating)) and float(v).is_integer() and abs(v) >= 1000):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _render(title, obj):
    # In Jupyter: a heading and an HTML table with scroll; elsewhere: the text block.
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
    except ImportError:
        return print(lines_es[-1] if LANG == "es" else lines[-1])
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(str(title))}</b></p>"
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        if isinstance(obj, pd.Series):          # rows are different quantities: each cell on its own
            df = obj.astype(object).map(_fmt).to_frame(name=translate("value", LANG))
        else:                                    # columns: the same number of decimals down each column
            df = obj.copy().astype(object)
            for k in range(df.shape[1]):
                col = obj.iloc[:, k]
                if pd.api.types.is_float_dtype(col) and not col.dropna().map(float.is_integer).all():
                    dec = max((len(f"{v:.4g}".split(".")[1]) if "." in f"{v:.4g}" and "e" not in f"{v:.4g}" else 0)
                              for v in col.dropna()) if col.notna().any() else 0
                    df.iloc[:, k] = col.map(lambda v, d=dec: "" if pd.isna(v) else f"{v:,.{d}f}")
                else:
                    df.iloc[:, k] = col.map(_fmt)
        table = df.to_html(na_rep="", border=0)
        _display(HTML(head + f"<div style='max-height:420px; overflow:auto'>{table}</div>"))
    else:
        _display(HTML(head + f"<pre style='margin:0'>{_html.escape(str(obj))}</pre>"))

def show(title, obj):
    # The English block is kept in `lines` (data/outputs_*.txt) and, in Spanish, the translated block in
    # `lines_es` (data/outputs_*_es.txt); on screen the table is shown as an HTML table.
    with contextlib.redirect_stdout(io.StringIO()):
        _show_en(title, obj)
        if LANG == "es":
            _show_en(translate(title, "es"), translate(obj, "es"))
            lines_es.append(lines.pop())
    _render(translate(title, LANG), translate(obj, LANG))
def is_unknown(s, field):
    """True where the field is unknown or not stated (blank counts as unknown)."""
    if field == "BMI":                       # stored as float: compare with a tolerance
        return s.isna() | (s.astype("Float64") > 99.85).fillna(False)
    if field in UNKNOWN:
        return s.isna() | s.isin(UNKNOWN[field])
    if isinstance(s.dtype, pd.CategoricalDtype) or s.dtype == object:
        return s.isna() | (s.astype("string") == CHAR_UNKNOWN)
    return s.isna()

print("cohorts:", COHORTS)

cohortes: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


## E4 · P1 · Auditoría de conteos y puerta de definición — población, eventos por etiqueta y reproducción de los conteos

Cuenta la población y las muertes por etiqueta en cada cohorte y las compara con los conteos de referencia del notebook 01.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Los conteos coinciden en las 8 cohortes | P1 pasa: la definición queda fija | Pasa |
| Alguna cohorte no coincide | Se detiene (`assert`): volver a E2 o E3 hasta que la definición sea estable | — |

In [2]:
# Q-N1 population and events by label; Q-N2 age at death in days; Q-N3 gestational-age bands;
# accumulators for Q-N4 to Q-N7 (whole population, never split by outcome)
pop_rows, age_rows, ga_rows, unk_year = [], [], [], []
cat_counts = {c: {} for c in CATEGORICAL}
num_counts = {c: {} for c in NUMERIC}
nunk_main, nunk_ge28 = {}, {}          # number of unknowns -> [records, neonatal deaths] (gate G2)
imputed = []
GA_BANDS = [(22, 23), (24, 27), (28, 31), (32, 36), (37, 47)]
for y in COHORTS:
    path = OUT_DIR / f"cohort_{y}.parquet"
    d = pd.read_parquet(path, columns=["RESTATUS", "COMBGEST", "COMBGST_IMP", "BWTIMP"] + LABELS + DEATH)
    keep = ((d["RESTATUS"] < 4) & d["COMBGEST"].between(22, 47)).to_numpy()
    d = d[keep]
    ge28 = d["COMBGEST"] >= 28
    pop_rows.append({"cohort": y, "population": len(d), "events_0_27d": int(d["label_main"].sum()),
                     "first_24h": int(d["died_first_24h"].sum()), "day1_27": int(d["label_day1_27"].sum()),
                     "early_0_6d": int(d["label_early_neonatal"].sum()),
                     "post_neonatal_28d_1y": int((d["died_infant"] & ~d["label_main"]).sum()),
                     "population_ge28": int(ge28.sum()), "events_ge28": int((ge28 & d["label_main"]).sum()),
                     "per_1000": round(1000 * d["label_main"].mean(), 2)})
    dead = d[d["label_main"]]
    a = dead["d_age_days"].astype("Int64")
    row = {"cohort": y, "under_1h": int((dead["d_ager5"] == 1).sum()), "1_23h": int((dead["d_ager5"] == 2).sum())}
    row.update({f"day_{k}": int((a == k).sum()) for k in range(1, 7)})
    row["day_7_27"] = int(a.between(7, 27).sum())
    row["age_missing"] = int(a.isna().sum())
    age_rows.append(row)
    for lo, hi in GA_BANDS:
        g = d["COMBGEST"].between(lo, hi)
        ga_rows.append({"cohort": y, "band": f"{lo}-{hi}", "population": int(g.sum()),
                        "events_0_27d": int((g & d["label_main"]).sum()),
                        "first_24h": int((g & d["died_first_24h"]).sum())})
    imputed.append({"cohort": y, "ga_imputed_pct": round(100 * (d["COMBGST_IMP"] == "1").mean(), 2),
                    "bw_imputed_pct": round(100 * (d["BWTIMP"] == "1").mean(), 2)})
    label = d["label_main"].to_numpy(bool)
    ge28 = ge28.to_numpy()
    del d, dead
    # one field at a time keeps the memory low (about 1 GB)
    n = np.zeros(int(keep.sum()), dtype=np.int16)
    row = {"cohort": y}
    for c in FIELDS:
        s = pd.read_parquet(path, columns=[c])[c][keep].reset_index(drop=True)
        u = is_unknown(s, c).to_numpy(bool)
        n += u
        row[c] = round(100 * u.mean(), 2)
        if c in CATEGORICAL:
            for k, v in s.astype("object").where(s.notna(), "missing").value_counts().items():
                cat_counts[c][k] = cat_counts[c].get(k, 0) + int(v)
        else:
            for k, v in s.astype("Float64").fillna(-1).value_counts().items():
                num_counts[c][round(float(k), 1)] = num_counts[c].get(round(float(k), 1), 0) + int(v)
        del s, u
    unk_year.append(row)
    for store, mask in ((nunk_main, np.ones(len(n), bool)), (nunk_ge28, ge28)):
        recs = np.bincount(n[mask]); evs = np.bincount(n[mask], weights=label[mask], minlength=len(recs))
        for k in np.nonzero(recs)[0]:
            r = store.setdefault(int(k), [0, 0])
            r[0] += int(recs[k]); r[1] += int(evs[k])
    del n, label, ge28, keep
    print(y, "done")

pop = pd.DataFrame(pop_rows).set_index("cohort")
show("Q-N1 Population (US residents, 22-47 weeks) and events by label", pop)
g1 = pd.Series({y: (int(pop.loc[y, "population"]), int(pop.loc[y, "events_0_27d"])) == EXPECTED[y]
                for y in COHORTS}, name="reproduced")
show("Gate G1: reference counts reproduced", g1)
assert g1.all(), "gate G1: the counts do not match the data adapter"

2016 lista
2017 lista
2018 lista
2019 lista
2020 lista
2021 lista
2022 lista
2023 lista


,población,eventos_0_27d,primeras_24h,día1_27,precoz_0_6d,posneonatal_28d_1a,población_28sem_o_más,eventos_28sem_o_más,por_1000
cohorte,,,,,,,,,
2016,"3,938,288","11,497","5,107","6,390","8,605","7,654","3,914,482","5,850",2.92
2017,"3,848,558","11,057","5,356","5,701","8,222","7,337","3,825,287","5,741",2.87
2018,"3,785,243","10,703","5,161","5,542","7,998","7,104","3,762,869","5,655",2.83
2019,"3,741,187","10,249","5,136","5,113","7,605","6,905","3,718,587","5,439",2.74
2020,"3,607,479","9,574","4,823","4,751","7,110","6,557","3,586,692","5,164",2.65
2021,"3,657,762","9,633","4,556","5,077","6,988","7,296","3,636,328","5,232",2.63
2022,"3,661,524","9,967","4,697","5,270","7,148","7,267","3,640,382","5,517",2.72
2023,"3,589,826","9,879","4,620","5,259","7,089","6,803","3,569,019","5,499",2.75


,valor
2016,Sí
2017,Sí
2018,Sí
2019,Sí
2020,Sí
2021,Sí
2022,Sí
2023,Sí


## E4 · Auditoría de conteos — edad al morir y bandas de edad gestacional

El punto de referencia de la sensibilidad (a) depende de cómo se codifica la edad al morir en los días 0 y 1; esta sección verifica que sea estable entre cohortes. La salida termina con la tabla «Decisión E4: punto de referencia…», que muestra el resultado de la revisión y el camino tomado. Le sigue «Decisión E4: edad gestacional». El criterio de lectura (cambio relativo de más de 20% en la proporción de muertes del día 1 entre la primera y la última cohorte) se agregó al reorganizar los notebooks, para que la decisión se vea en la salida; la decisión misma se tomó antes de ajustar (ficha del desenlace).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Muertes del día 1 estables entre cohortes | Se mantiene el punto de referencia a las 24 h | — |
| Cambian entre cohortes | Se mueve el punto de referencia a una definición que no dependa de ese cambio, registrado antes de ajustar | Día 1: 1.412 (2016) → ~430 (2019+): **M4-D1, 48 h** |

In [3]:
# Q-N2 age at death of the neonatal deaths: hours on day 0, then days
age = pd.DataFrame(age_rows).set_index("cohort")
show("Q-N2 Neonatal deaths by age at death (day 0 split into <1 h and 1-23 h)", age)
share = pd.DataFrame({"day0_pct": (100 * (age["under_1h"] + age["1_23h"]) / pop["events_0_27d"]).round(1),
                      "day1_pct": (100 * age["day_1"] / pop["events_0_27d"]).round(1),
                      "day0_plus_day1_pct": (100 * (age["under_1h"] + age["1_23h"] + age["day_1"])
                                             / pop["events_0_27d"]).round(1),
                      "day2_27_pct": (100 * (pop["events_0_27d"] - age["under_1h"] - age["1_23h"] - age["day_1"])
                                      / pop["events_0_27d"]).round(1)})
show("Q-N2b Share of the neonatal deaths: day 0, day 1, days 0-1 together and days 2-27", share)

# Q-N3 gestational-age bands, all cohorts together
ga = pd.DataFrame(ga_rows).groupby("band", sort=False)[["population", "events_0_27d", "first_24h"]].sum()
ga["pct_population"] = (100 * ga["population"] / ga["population"].sum()).round(2)
ga["pct_events"] = (100 * ga["events_0_27d"] / ga["events_0_27d"].sum()).round(1)
ga["per_1000"] = (1000 * ga["events_0_27d"] / ga["population"]).round(1)
show("Q-N3 Gestational-age bands (combined gestation), cohorts 2016-2023", ga)
show("Q-N3b Gestational age and birthweight imputed by NCHS (% of the population)",
     pd.DataFrame(imputed).set_index("cohort"))

# Decision of this section (M4-D1): is the coding of deaths on day 0 and day 1 stable across cohorts?
# Reading criterion: the share of the neonatal deaths that falls on day 1 changes by more than 20% (relative)
# between the first and the last cohort. Shares are used, not counts, because neonatal deaths also fall over time.
pd.set_option("display.max_colwidth", None)
d1 = age["day_1"]
rel = lambda s: 100 * (s.iloc[-1] - s.iloc[0]) / s.iloc[0]
stable = abs(rel(share["day1_pct"])) <= 20
show("S4 decision: landmark of sensitivity analysis (a) (M4-D1)", pd.Series({
    "day-1 deaths, first vs last cohort": f"{int(d1.iloc[0])} vs {int(d1.iloc[-1])}",
    "% of the deaths on day 1, first vs last cohort": f"{share['day1_pct'].iloc[0]} vs {share['day1_pct'].iloc[-1]}",
    "relative change of that share (%)": round(rel(share["day1_pct"]), 1),
    "% of the deaths on days 0-1 together, first vs last cohort": f"{share['day0_plus_day1_pct'].iloc[0]} vs {share['day0_plus_day1_pct'].iloc[-1]}",
    "relative change of that share (%) ": round(rel(share["day0_plus_day1_pct"]), 1),
    "day-1 coding stable (change within 20%)": bool(stable),
    "decision": "24-hour landmark kept" if stable else
                "M4-D1: landmark moved to 48 hours (days 0-1 together do not depend on how day 0 and day 1 are split)"}))

# What the gestational-age bands lead to
ga_short = ga.loc[["22-23", "24-27"]]
imp = pd.DataFrame(imputed).set_index("cohort")
show("S4 decision: gestational age", pd.Series({
    "% of births at 22-27 weeks": round(ga_short["pct_population"].sum(), 2),
    "% of deaths at 22-27 weeks": round(100 * ga_short["events_0_27d"].sum() / ga["events_0_27d"].sum(), 1),
    "consequence": "sensitivity analysis (c) restricted to 28 weeks or more (outcome card)",
    "gestational age imputed by NCHS, highest cohort (%)": float(imp["ga_imputed_pct"].max()),
    "consequence for the imputed values": "kept as recorded; the imputation flags stay as metadata, not as predictors"}))

,menos_1h,1_23h,día_1,día_2,día_3,día_4,día_5,día_6,días_7_27,edad_faltante
cohorte,,,,,,,,,,
2016,"1,823","3,284","1,412",722,446,350,303,265,"2,892",0
2017,"1,648","3,708",845,725,444,309,279,264,"2,835",0
2018,"1,669","3,492",802,738,428,350,279,240,"2,705",0
2019,"1,598","3,538",431,714,440,346,265,273,"2,644",0
2020,"1,515","3,308",404,664,429,296,265,229,"2,464",0
2021,"1,322","3,234",433,672,452,334,286,255,"2,645",0
2022,"1,395","3,302",440,676,427,344,277,287,"2,819",0
2023,"1,346","3,274",425,729,451,319,279,266,"2,790",0


,día0_%,día1_%,día0_más_día1_%,días2_27_%
cohorte,,,,
2016,44.4,12.3,56.7,43.3
2017,48.4,7.6,56.1,43.9
2018,48.2,7.5,55.7,44.3
2019,50.1,4.2,54.3,45.7
2020,50.4,4.2,54.6,45.4
2021,47.3,4.5,51.8,48.2
2022,47.1,4.4,51.5,48.5
2023,46.8,4.3,51.1,48.9


,población,eventos_0_27d,primeras_24h,%_población,%_eventos,por_1000
banda,,,,,,
22-23,"34,964","18,843","13,042",0.12,22.8,538.9
24-27,"141,257","19,619","8,189",0.47,23.8,138.9
28-31,"355,899","9,416","4,594",1.19,11.4,26.5
32-36,"3,003,558","13,908","6,734",10.07,16.8,4.6
37-47,"26,294,189","20,773","6,897",88.15,25.2,0.8


,EG_imputada_%,peso_imputado_%
cohorte,,
2016,2.02,0.07
2017,1.86,0.00
2018,1.73,0.01
2019,1.54,0.01
2020,1.51,0.01
2021,1.43,0.04
2022,1.38,0.04
2023,1.49,0.03


,valor
"muertes del día 1, primera vs última cohorte",1412 vs 425
"% de las muertes en el día 1, primera vs última cohorte",12.3 vs 4.3
cambio relativo de esa proporción (%),-65
"% de las muertes en los días 0-1 juntos, primera vs última cohorte",56.7 vs 51.1
cambio relativo de esa proporción (%),-9.9
codificación del día 1 estable (cambio dentro de 20%),No
decisión,M4-D1: el punto de referencia pasa a las 48 horas (los días 0-1 juntos no dependen de cómo se separen el día 0 y el día 1)


,valor
% de nacimientos a las 22-27 semanas,0.59
% de muertes a las 22-27 semanas,46.6
consecuencia,análisis de sensibilidad (c) restringido a 28 semanas o más (ficha del desenlace)
"edad gestacional imputada por el NCHS, cohorte más alta (%)",2.02
consecuencia para los valores imputados,"se usan como están registrados; las marcas de imputación quedan como metadatos, no como predictores"


## E4 · Auditoría de conteos — frecuencias de los predictores candidatos

Frecuencias y percentiles de cada candidato en toda la población, nunca separados por desenlace, y % de desconocidos por cohorte. La salida termina con la tabla «Decisión E4: candidatos con más de 5% de desconocidos (M4-D2)», que muestra el resultado de la revisión y el camino tomado.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Candidato con más de 5% de desconocidos o con un cambio de reporte entre cohortes | Se decide antes de E7 si entra, con qué tratamiento, o se excluye | Estado civil (0% en 2016, ~12% desde 2017): **M4-D2, excluido**. Intervalo entre embarazos (11,6%): entra |
| Resto de candidatos (≤ 3% de desconocidos) | Entran a E7 | Sí |

In [4]:
# Q-N4 frequencies of the categorical candidates (whole population, never split by outcome)
total = int(pop["population"].sum())
for block, cols in BLOCKS.items():
    for c in cols:
        if c in CATEGORICAL:
            s = pd.Series(cat_counts[c])
            s = s.reindex(sorted(s.index, key=str))
            show(f"Q-N4 [{block}] {c}: births and % of the population",
                 pd.DataFrame({"n": s, "pct": (100 * s / total).round(2)}))

# Q-N5 numeric candidates: unknown and percentiles (unknown codes and the 888 'not applicable' excluded)
rows = []
for c in NUMERIC:
    s = pd.Series(num_counts[c])
    excl = set(UNKNOWN.get(c, [])) | {-1.0} | ({888.0} if c == "ILP_R" else set())
    unk = s[[k for k in s.index if k in excl and k != 888.0]].sum()
    v = s[[k for k in s.index if k not in excl and not (c == "BMI" and abs(k - 99.9) < 1e-3)]].sort_index()
    cum = v.cumsum() / v.sum()
    row = {"field": c, "pct_unknown": round(100 * unk / total, 2)}
    row.update({f"p{int(q * 100):02d}": float(v.index[np.searchsorted(cum.values, q)])
                for q in (0.05, 0.25, 0.5, 0.75, 0.95)})
    if c == "ILP_R":
        row["pct_not_applicable_888"] = round(100 * s.get(888.0, 0) / total, 2)
    rows.append(row)
show("Q-N5 Numeric candidates: % unknown and percentiles", pd.DataFrame(rows).set_index("field"))

# Q-N6 % unknown of every candidate by cohort (whole population)
uy = pd.DataFrame(unk_year).set_index("cohort").T
uy["all_cohorts"] = (uy.mul(pop["population"], axis=1).sum(axis=1) / total).round(2)
show("Q-N6 % unknown by cohort, whole population", uy)
show("Q-N6b Candidates with more than 5% unknown in any cohort", uy[uy.drop(columns="all_cohorts").max(axis=1) > 5])

# Decision of this section (M4-D2): candidates with more than 5% unknown in any cohort.
# A field reported by every state in some cohorts (under 1% unknown) and missing in others (over 5%) has a
# reporting change: its unknowns depend on the cohort, not on the birth.
by_year = uy.drop(columns="all_cohorts")
high = by_year[by_year.max(axis=1) > 5].index
dec = {}
for f in high:
    reporting_change = bool(by_year.loc[f].min() < 1)
    dec[f] = {"% unknown, all cohorts": float(uy.loc[f, "all_cohorts"]),
              "% unknown, lowest-highest cohort": f"{by_year.loc[f].min():.1f}-{by_year.loc[f].max():.1f}",
              "reporting change across cohorts": reporting_change,
              "decision": "M4-D2: excluded" if reporting_change else "enters S7 (unknowns imputed)"}
others = uy.drop(index=high)
dec["other candidates"] = {"% unknown, all cohorts": float(others["all_cohorts"].max()),
                           "% unknown, lowest-highest cohort": f"up to {others.drop(columns='all_cohorts').max().max():.1f}",
                           "reporting change across cohorts": False, "decision": "enter S7"}
show("S4 decision: candidates with more than 5% unknown (M4-D2)", pd.DataFrame(dec).T)

,n,%
1,"15,262,446",51.16
2,"4,255,917",14.27
3,"221,505",0.74
4,"1,849,025",6.20
5,"77,177",0.26
6,"678,841",2.28
7,"7,214,671",24.19
8,"270,285",0.91


,n,%
1,"942,574",3.16
2,"2,645,563",8.87
3,"7,671,576",25.72
4,"5,677,007",19.03
5,"2,465,372",8.26
6,"6,233,638",20.90
7,"2,913,660",9.77
8,"850,953",2.85
9,"429,524",1.44


,n,%
1,"16,046,582",53.79
2,"10,755,284",36.06
faltante,"3,028,001",10.15


,n,%
1,"23,028,629",77.2
2,"6,740,864",22.6
3,"60,374",0.2


,n,%
1,"12,437,477",41.69
2,"14,918,763",50.01
3,"1,213,343",4.07
4,"1,058,875",3.55
9,"201,409",0.68


,n,%
N,"19,454,936",65.22
U,"332,882",1.12
Y,"10,042,049",33.66


... (39 outputs left out)

,%_desconocido,p05,p25,p50,p75,p95,%_no_aplica_888
campo,,,,,,,
MAGER,0.00,20.0,25.0,29.0,33,39.0,
PRIORLIVE,0.15,0.0,0.0,1.0,2,4.0,
PRIORDEAD,0.26,0.0,0.0,0.0,0,0.0,
PRIORTERM,0.30,0.0,0.0,0.0,1,2.0,
ILP_R,11.55,10.0,19.0,31.0,54,117.0,30.99
RF_CESARN,0.13,0.0,0.0,0.0,0,1.0,
PRECARE,2.26,1.0,2.0,3.0,3,6.0,
PREVIS,2.30,4.0,9.0,11.0,13,17.0,
BMI,2.23,19.1,22.4,25.8,31,40.7,


cohorte,2016,2017,2018,2019,2020,2021,2022,2023,todas_las_cohortes
MAGER,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
MRACEHISP,0.90,0.83,0.87,0.94,0.92,0.97,0.93,0.89,0.91
MEDUC,1.28,1.26,1.27,1.42,1.41,1.51,1.64,1.76,1.44
DMAR,0.00,12.24,12.00,11.91,11.63,11.48,11.43,11.12,10.15
MBSTATE_REC,0.20,0.19,0.19,0.20,0.19,0.21,0.22,0.22,0.20
PAY_REC,0.59,0.58,0.57,0.67,0.69,0.73,0.87,0.71,0.67
WIC,1.23,1.14,1.09,1.13,1.01,1.09,1.13,1.09,1.12
PRIORLIVE,0.18,0.14,0.18,0.14,0.12,0.16,0.17,0.13,0.15
PRIORDEAD,0.36,0.21,0.25,0.20,0.19,0.29,0.29,0.24,0.25
PRIORTERM,0.46,0.27,0.31,0.23,0.22,0.30,0.30,0.28,0.30


cohorte,2016,2017,2018,2019,2020,2021,2022,2023,todas_las_cohortes
DMAR,0.00,12.24,12.00,11.91,11.63,11.48,11.43,11.12,10.15
ILP_R,12.41,11.96,12.01,11.92,11.42,11.06,10.95,10.55,11.55


,"% desconocido, todas las cohortes","% desconocido, cohorte más baja-más alta",cambio de reporte entre cohortes,decisión
DMAR,10.15,0.0-12.2,Sí,M4-D2: excluido
ILP_R,11.55,10.6-12.4,No,entra a E7 (desconocidos imputados)
otros candidatos,2.97,hasta 3.6,No,entran a E7


## P2 · Puerta de fuga (verificación previa) — ¿el número de desconocidos por sí solo separa las muertes?

Si los desconocidos se concentran en los eventos, un modelo podría aprender cómo se llenó el certificado en vez del riesgo. La salida termina con la tabla «Decisión previa de P2: manejo de los desconocidos (M4-D3)», que muestra el resultado de la revisión y el camino tomado. El criterio de lectura (AUROC mayor que 0,55) se agregó al reorganizar los notebooks, para que la decisión se vea en la salida; la decisión misma se tomó antes de ajustar (ficha del desenlace).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| AUROC del número de desconocidos ≈ 0,50 | Sin tratamiento especial | — |
| Claramente por encima de 0,50 | Se fija antes de ajustar: imputación sin indicadores de faltante y sensibilidad con registros completos | 0,577 (0,556 en ≥ 28 semanas): **M4-D3** |

In [5]:
# Q-N7 leakage check (gate G2): does the number of unknown fields, by itself, separate the deaths?
def auroc_from_counts(table):
    """AUROC of an integer score from a {score: [records, events]} table (ties count one half)."""
    t = pd.DataFrame(table, index=["records", "events"]).T.sort_index()
    ev = t["events"].to_numpy(float); ne = (t["records"] - t["events"]).to_numpy(float)
    below = np.concatenate([[0], np.cumsum(ne)[:-1]])
    return float((ev * (below + 0.5 * ne)).sum() / (ev.sum() * ne.sum()))

def summary(table):
    t = pd.DataFrame(table, index=["records", "events"]).T.sort_index()
    t.index = [str(k) if k < 5 else "5+" for k in t.index]
    return t.groupby(level=0, sort=False).sum()

g2 = pd.DataFrame({"population": [auroc_from_counts(nunk_main)], "ge28_weeks": [auroc_from_counts(nunk_ge28)]},
                  index=["AUROC of the number of unknowns"]).round(3)
show("Q-N7 Gate G2: AUROC of the number of unknown fields alone (neonatal death 0-27 days)", g2)
show("Q-N7b Number of unknown fields: births (whole population)", summary(nunk_main)[["records"]])

# Preliminary G2 decision (M4-D3): does the number of unknowns alone separate the deaths?
# Reading criterion: an AUROC above 0.55 counts as clearly above chance (0.50).
auc_pop, auc_28 = float(g2.loc["AUROC of the number of unknowns", "population"]), float(g2.loc["AUROC of the number of unknowns", "ge28_weeks"])
above = auc_pop > 0.55
show("Preliminary G2 decision: handling of unknowns (M4-D3)", pd.Series({
    "AUROC of the number of unknowns (population / 28 weeks or more)": f"{auc_pop:.3f} / {auc_28:.3f}",
    "clearly above 0.50 (AUROC > 0.55)": bool(above),
    "decision": "M4-D3: imputation without missingness indicators, plus a complete-record sensitivity analysis" if above
                else "no special handling of unknowns"}))

write_outputs(OUT_FILE, "Count audit of the neonatal population (aggregate counts only; 1-9 shown as <10)")
print("written:", OUT_FILE)

,población,28_semanas_o_más
AUROC del número de desconocidos,0.577,0.556


,registros
0,"21,579,174"
1,"6,210,424"
2,"1,197,407"
3,"439,077"
4,"145,268"
5+,"258,517"


,valor
AUROC del número de desconocidos (población / 28 semanas o más),0.577 / 0.556
claramente por encima de 0.50 (AUROC > 0.55),Sí
decisión,"M4-D3: imputación sin indicadores de faltante, más un análisis de sensibilidad con registros completos"


escrito: d:\Emmanuel\Uninorte\Tesis\NeonatalMortality_NCHS\data\outputs_02_count_audit.txt


## Resumen del notebook

Indicadores principales de la auditoría.

In [6]:
# Summary of notebook 02
ga_ = ga.copy()
summary = pd.Series({
    "population (US residents, 22-47 weeks)": int(pop["population"].sum()),
    "neonatal deaths 0-27 days": int(pop["events_0_27d"].sum()),
    "gate G1 (reference counts reproduced)": "passes" if bool(g1.all()) else "DOES NOT PASS",
    "% of deaths at 22-27 weeks": round(100 * ga_.loc[["22-23", "24-27"], "events_0_27d"].sum() / ga_["events_0_27d"].sum(), 1),
    "% of births at 22-27 weeks": round(ga_.loc[["22-23", "24-27"], "pct_population"].sum(), 2),
    "day-1 deaths, first vs last cohort": f"{int(age['day_1'].iloc[0])} vs {int(age['day_1'].iloc[-1])}",
    "candidates with more than 5% unknown in any cohort": ", ".join(uy[uy.drop(columns='all_cohorts').max(axis=1) > 5].index) or "none",
    "AUROC of the number of unknowns alone": float(g2.loc["AUROC of the number of unknowns", "population"])}, name="value")
translate(summary.to_frame(), LANG)

,valor
"población (residentes en EE. UU., 22-47 semanas)",29829867
muertes neonatales 0-27 días,82559
puerta P1 (conteos de referencia reproducidos),cumple
% de muertes a las 22-27 semanas,46.6
% de nacimientos a las 22-27 semanas,0.59
"muertes del día 1, primera vs última cohorte",1412 vs 425
candidatos con más del 5% desconocido en alguna cohorte,"DMAR, ILP_R"
AUROC del número de desconocidos solo,0.577
